In [19]:
import re
import shlex
import os
from urllib.parse import unquote

In [25]:
def parse_curl_cookies(curl_command):
    """Parses the curl command to extract headers and cookies."""
    from urllib.parse import unquote
    import shlex
    
    # Split the command using shlex to handle quotes correctly
    try:
        args = shlex.split(curl_command)
    except ValueError as e:
        print(f"Error parsing curl command: {e}")
        return None

    headers = {}
    cookies = {}
    
    # Iterate through arguments to find headers and cookies
    for i in range(len(args)):
        arg = args[i]
        
        # Check for -H or --header
        if arg in ('-H', '--header') and i + 1 < len(args):
            header_parts = args[i+1].split(':', 1)
            if len(header_parts) == 2:
                key = header_parts[0].strip().lower()
                value = header_parts[1].strip()
                headers[key] = value
        
        # Check for -b or --cookie (usually in curl format as 'key=value; key2=value2')
        # Note: sometimes cookies are passed as a header 'cookie: ...'
        elif arg in ('-b', '--cookie') and i + 1 < len(args):
            cookie_str = args[i+1]
            for part in cookie_str.split(';'):
                if '=' in part:
                    k, v = part.strip().split('=', 1)
                    cookies[k] = v

    # Also separate Authorization header if present
    bearer_token = ""
    # Check specifically for case-insensitive 'authorization'
    auth_header_key = next((k for k in headers if k.lower() == 'authorization'), None)
    
    if auth_header_key:
        auth_val = headers[auth_header_key]
        # Handle both 'Bearer TOKEN' and just 'TOKEN' if necessary, but usually it's Bearer
        if auth_val.lower().startswith("bearer"):
            # Split by space to handle multiple spaces safely
            parts = auth_val.split(maxsplit=1)
            if len(parts) == 2:
                bearer_token = parts[1].strip()
            else:
                # Fallback if weird formatting
                bearer_token = auth_val[7:].strip()
        else:
            # Maybe it's just the token?
            bearer_token = auth_val.strip()
            
        if '%' in bearer_token:
            bearer_token = unquote(bearer_token)

    csrf_token = headers.get('x-csrf-token', '')
    
    if 'cookie' in headers:
        cookie_str = headers['cookie']
        for part in cookie_str.split(';'):
            if '=' in part:
                k, v = part.strip().split('=', 1)
                cookies[k] = v

    auth_token_cookie = cookies.get('auth_token', '')
    ct0_cookie = cookies.get('ct0', '')
    guest_id = cookies.get('guest_id', '')
    
    final_csrf = csrf_token if csrf_token else ct0_cookie

    return {
        "BEARER_TOKEN": bearer_token,
        "CSRF_TOKEN": final_csrf,
        "AUTH_TOKEN": auth_token_cookie,
        "GUEST_ID": guest_id
    }

def print_env_variables(extracted_data, username):
    """Prints the env variables in the correct format."""
    suffix = f"_{username.lower()}"
    print("\n# Add these lines to your .env file:\n")
    for key, value in extracted_data.items():
        if value:
             print(f"{key}{suffix}=\"{value}\"")
    print("\n")

In [34]:
# PASTE YOUR CURL COMMAND HERE between the quotes
curl_command = r"""

"""

# DEFINE THE USERNAME HERE
username = "pussy"

In [35]:
if curl_command.strip():
    data = parse_curl_cookies(curl_command)
    if data:
        print(f"Extracted Data for user: {username}\n")
        print_env_variables(data, username)
    else:
        print("Failed to extract data.")
else:
    print("Please paste a curl command in the cell above.")

Please paste a curl command in the cell above.
